In [1]:
# Tìm thư mục gốc để notebook chạy được từ notebooks/ hoặc thư mục project.
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir()
)
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / 'data'
MODEL_DIR = project_root / 'models'
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

In [2]:
from src.models.fine_tuning import build_model, fine_tune_model

In [3]:
train_df = pd.read_csv(DATA_DIR / 'train_split.csv')
val_df = pd.read_csv(DATA_DIR / 'validation_split.csv')
test_df = pd.read_csv(DATA_DIR / 'test_split.csv')

# Ảnh được đổi về cùng kích thước và chuẩn hóa giống pipeline hiện tại.
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    paths = [str(DATA_DIR / name) for name in frame['file_name']]
    labels = frame['label'].to_numpy(dtype='float32')
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(1000)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(16).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
y_test = test_df['label'].to_numpy()

In [ ]:
model = build_model()

# Pha 1: backbone đang đóng băng, chỉ học bộ phân loại mới.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-6
)
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

In [ ]:
# Pha 2: mở một phần MobileNet và tiếp tục học từ trọng số pha 1.
# Learning rate nhỏ giúp giữ lại kiến thức pretrained đã có.
model = fine_tune_model(model)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-7
)
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

In [ ]:
# Chỉ dùng tập test sau khi đã hoàn tất huấn luyện.
test_loss, test_accuracy, test_auc = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f}; AUC: {test_auc:.4f}')

probabilities = model.predict(test_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype('int32')
print(classification_report(
    y_test, predictions, target_names=['Real', 'AI-generated'], digits=4
))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / 'fine_tuned_mobilenetv2.keras')